In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Assuming 'epochs' is your MNE Epochs object
shape_of_data = epochs.get_data().shape
print(shape_of_data)

In [ ]:
# ============================================================
# EXTRACT SPECTRAL COHERENCE FEATURES FOR ASD & TD GROUPS
# (FINAL FIX: Removing 'output="dense"', which was the cause of the
# "all zeros" bug, and copying the user's own working .get_data() method)
# ============================================================

import os
import numpy as np
import pandas as pd
import mne
from mne_connectivity import spectral_connectivity_epochs

# ----------- PATH SETUP -----------
base_path = "/kaggle/input/preprocessed-small-autism-dataset/preprocessed"
output_dir = "/kaggle/working/coherence_features"
os.makedirs(output_dir, exist_ok=True)

# ----------- PARAMETERS -----------
groups = ["ASD", "TD"]
freq_bands = {
    "delta": (1.25, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}
method = "coh"  # coherence method
band_names = list(freq_bands.keys())

# ----------- PROCESS FUNCTION -----------
def extract_coherence_features(group_name):
    group_path = os.path.join(base_path, group_name)
    fif_files = sorted([f for f in os.listdir(group_path) if f.endswith("-epo.fif")])

    features_list = []
    subject_ids = []
    column_names = None  # To store column names after first subject

    print(f"\n🔹 Processing group: {group_name}")
    for file in fif_files:
        subject_id = file.split("_")[0]
        file_path = os.path.join(group_path, file)
        print(f"  → Computing coherence for subject {subject_id}")

        try:
            # --- Load epochs ---
            epochs = mne.read_epochs(file_path, preload=True, verbose=False)
            ch_names = epochs.info["ch_names"]
            n_channels = len(ch_names)
            indices = np.triu_indices(n_channels, k=1) # <-- This is already here, perfect.

            # --- Generate Column Names (only once) ---
            if column_names is None:
                print("    - Generating feature names (first subject only)...")
                column_names = []
                for (idx1, idx2) in zip(indices[0], indices[1]):
                    pair_name = f"{ch_names[idx1]}-{ch_names[idx2]}"
                    for band_name in band_names:
                        column_names.append(f"{band_name}_{pair_name}")
                print(f"    - Generated {len(column_names)} feature names.")
            
            # --- Loop through each band one-by-one ---
            all_band_features = [] # To store features for each band
            
            for band_name, (fmin, fmax) in freq_bands.items():
                print(f"      - Computing {band_name} band ({fmin}-{fmax} Hz)...")
                con = spectral_connectivity_epochs(
                    epochs,
                    indices=indices, # <--- THIS IS THE FIX. Tell the function to only compute the upper triangle.
                    method=method,
                    mode='multitaper',
                    fmin=fmin,
                    fmax=fmax,
                    sfreq=epochs.info['sfreq'],
                    faverage=True,
                    n_jobs=-1,
                    verbose=False,
                )
                
                # --- THIS IS THE FIX ---
                # REMOVED: output="dense", which was causing the all-zeros bug.
                # We now get the data as a (n_pairs, 1) array, just like
                # the user's original working code.
                
                # con.get_data() returns (n_pairs, n_freqs)
                # Since faverage=True, n_freqs=1. We take [:, 0]
                # Shape becomes (n_pairs,)
                con_data_upper = con.get_data()[:, 0]
                all_band_features.append(con_data_upper)

            # --- Flatten all features ---
            # Stack arrays: shape (n_bands, n_pairs) -> (5, 1953)
            stacked_features = np.stack(all_band_features, axis=0)
            
            # Transpose: shape (n_pairs, n_bands) -> (1953, 5)
            stacked_features_transposed = stacked_features.T
            
            # Ravel: Flattens to (pair1_delta, pair1_theta, ..., pair2_delta, ...)
            features_flat = stacked_features_transposed.ravel()

            # --- DEBUG CHECK: Verify non-zero values ---
            mean_val = np.mean(features_flat)
            is_all_zero = np.all(features_flat == 0)
            print(f"    - [Debug Check] Mean coherence: {mean_val:.4f} | All zeros: {is_all_zero}")
            # --- End Debug Check ---
            
            features_list.append(features_flat)
            subject_ids.append(f"{group_name}_{subject_id}")

        except Exception as e:
            print(f"  ❌ ERROR processing subject {subject_id}: {e}")

    # --- Combine into DataFrame ---
    if not features_list:
        print(f"\nNo features were extracted for group {group_name}.")
        return

    # Create DataFrame with the correct column names and subject IDs
    features_df = pd.DataFrame(features_list, columns=column_names)
    features_df.insert(0, "subject_id", subject_ids)

    # --- Save to CSV ---
    save_path = os.path.join(output_dir, f"{group_name}_coherence_features.csv")
    features_df.to_csv(save_path, index=False)
    print(f"✅ Saved {group_name} features to {save_path}")
    print(f"    → Shape: {features_df.shape}")

# ----------- RUN FOR BOTH GROUPS -----------
for grp in groups:
    extract_coherence_features(grp)

print("\n🎯 All coherence features extracted and saved successfully.")




In [3]:
# ============================================================
# EXTRACT SPECTRAL IMAGINARY COHERENCY (imcoh) FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd
import mne
from mne_connectivity import spectral_connectivity_epochs

# ----------- PATH SETUP -----------
base_path = "/kaggle/input/preprocessed-small-autism-dataset/preprocessed"
# MODIFIED: Changed output directory name
output_dir = "/kaggle/working/imcoh_features"
os.makedirs(output_dir, exist_ok=True)

# ----------- PARAMETERS -----------
groups = ["ASD", "TD"]
freq_bands = {
    "delta": (1.25, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}
# --- MODIFIED: Changed method from 'coh' to 'imcoh' ---
method = "imcoh"  # Imaginary Coherency
band_names = list(freq_bands.keys())

# ----------- PROCESS FUNCTION -----------
# MODIFIED: Renamed function
def extract_imcoh_features(group_name):
    group_path = os.path.join(base_path, group_name)
    fif_files = sorted([f for f in os.listdir(group_path) if f.endswith("-epo.fif")])

    features_list = []
    subject_ids = []
    column_names = None  # To store column names after first subject

    print(f"\n🔹 Processing group: {group_name}")
    for file in fif_files:
        subject_id = file.split("_")[0]
        file_path = os.path.join(group_path, file)
        # MODIFIED: Updated print statement
        print(f"  → Computing Imaginary Coherency (imcoh) for subject {subject_id}")

        try:
            # --- Load epochs ---
            epochs = mne.read_epochs(file_path, preload=True, verbose=False)
            ch_names = epochs.info["ch_names"]
            n_channels = len(ch_names)
            indices = np.triu_indices(n_channels, k=1) 

            # --- Generate Column Names (only once) ---
            if column_names is None:
                print("    - Generating feature names (first subject only)...")
                column_names = []
                for (idx1, idx2) in zip(indices[0], indices[1]):
                    pair_name = f"{ch_names[idx1]}-{ch_names[idx2]}"
                    for band_name in band_names:
                        column_names.append(f"{band_name}_{pair_name}")
                print(f"    - Generated {len(column_names)} feature names.")
            
            # --- Loop through each band one-by-one ---
            all_band_features = [] # To store features for each band
            
            for band_name, (fmin, fmax) in freq_bands.items():
                print(f"      - Computing {band_name} band ({fmin}-{fmax} Hz)...")
                con = spectral_connectivity_epochs(
                    epochs,
                    indices=indices, # Only compute the upper triangle
                    method=method,   # This will now use "imcoh"
                    mode='multitaper',
                    fmin=fmin,
                    fmax=fmax,
                    sfreq=epochs.info['sfreq'],
                    faverage=True,
                    n_jobs=-1,
                    verbose=False,
                )
                
                # --- Get data without output="dense" ---
                # This returns (n_pairs, 1), so we take [:, 0]
                con_data_upper = con.get_data()[:, 0]
                all_band_features.append(con_data_upper)

            # --- Flatten all features ---
            stacked_features = np.stack(all_band_features, axis=0)
            stacked_features_transposed = stacked_features.T
            features_flat = stacked_features_transposed.ravel()

            # --- DEBUG CHECK: Verify non-zero values ---
            mean_val = np.mean(features_flat)
            is_all_zero = np.all(features_flat == 0)
            # MODIFIED: Updated print statement
            print(f"    - [Debug Check] Mean imcoh: {mean_val:.4f} | All zeros: {is_all_zero}")
            # --- End Debug Check ---
            
            features_list.append(features_flat)
            subject_ids.append(f"{group_name}_{subject_id}")

        except Exception as e:
            print(f"  ❌ ERROR processing subject {subject_id}: {e}")

    # --- Combine into DataFrame ---
    if not features_list:
        print(f"\nNo features were extracted for group {group_name}.")
        return

    # Create DataFrame with the correct column names and subject IDs
    features_df = pd.DataFrame(features_list, columns=column_names)
    features_df.insert(0, "subject_id", subject_ids)

    # --- Save to CSV ---
    # MODIFIED: Updated save path
    save_path = os.path.join(output_dir, f"{group_name}_imcoh_features.csv")
    features_df.to_csv(save_path, index=False)
    print(f"✅ Saved {group_name} features to {save_path}")
    print(f"    → Shape: {features_df.shape}")

# ----------- RUN FOR BOTH GROUPS -----------
# MODIFIED: Calls renamed function
for grp in groups:
    extract_imcoh_features(grp)

# MODIFIED: Updated print statement
print("\n🎯 All Imaginary Coherency features extracted and saved successfully.")



🔹 Processing group: ASD
  → Computing Imaginary Coherency (imcoh) for subject 001
    - Generating feature names (first subject only)...
    - Generated 9765 feature names.
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean imcoh: -0.0003 | All zeros: False
  → Computing Imaginary Coherency (imcoh) for subject 002
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean imcoh: -0.0002 | All zeros: False
  → Computing Imaginary Coherency (imcoh) for subject 003
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...

In [2]:
!pip install mne_connectivity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 2.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 78.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.2 MB/s eta 0:00:0000:01:00:01


In [3]:
# ============================================================
# EXTRACT SPECTRAL PHASE LOCKING VALUE (PLV) FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd
import mne
from mne_connectivity import spectral_connectivity_epochs

# ----------- PATH SETUP -----------
base_path = "/kaggle/input/preprocessed-small-autism-dataset/preprocessed"
# MODIFIED: Changed output directory name for PLV features
output_dir = "/kaggle/working/plv_features"
os.makedirs(output_dir, exist_ok=True)

# ----------- PARAMETERS -----------
groups = ["ASD", "TD"]
freq_bands = {
    "delta": (1.25, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}
# --- MODIFIED: Changed method from 'imcoh' to 'plv' ---
method = "plv"  # Phase Locking Value
band_names = list(freq_bands.keys())

# ----------- PROCESS FUNCTION -----------
# MODIFIED: Renamed function for clarity
def extract_plv_features(group_name):
    group_path = os.path.join(base_path, group_name)
    fif_files = sorted([f for f in os.listdir(group_path) if f.endswith("-epo.fif")])

    features_list = []
    subject_ids = []
    column_names = None  # To store column names after first subject

    print(f"\n🔹 Processing group: {group_name}")
    for file in fif_files:
        subject_id = file.split("_")[0]
        file_path = os.path.join(group_path, file)
        # MODIFIED: Updated print statement for PLV
        print(f"  → Computing Phase Locking Value (PLV) for subject {subject_id}")

        try:
            # --- Load epochs ---
            epochs = mne.read_epochs(file_path, preload=True, verbose=False)
            ch_names = epochs.info["ch_names"]
            n_channels = len(ch_names)
            indices = np.triu_indices(n_channels, k=1) 

            # --- Generate Column Names (only once) ---
            if column_names is None:
                print("    - Generating feature names (first subject only)...")
                column_names = []
                for (idx1, idx2) in zip(indices[0], indices[1]):
                    pair_name = f"{ch_names[idx1]}-{ch_names[idx2]}"
                    for band_name in band_names:
                        column_names.append(f"{band_name}_{pair_name}")
                print(f"    - Generated {len(column_names)} feature names.")
            
            # --- Loop through each band one-by-one ---
            all_band_features = [] # To store features for each band
            
            for band_name, (fmin, fmax) in freq_bands.items():
                print(f"      - Computing {band_name} band ({fmin}-{fmax} Hz)...")
                con = spectral_connectivity_epochs(
                    epochs,
                    indices=indices, # Only compute the upper triangle
                    method=method,   # This will now use "plv"
                    mode='multitaper',
                    fmin=fmin,
                    fmax=fmax,
                    sfreq=epochs.info['sfreq'],
                    faverage=True,
                    n_jobs=-1,
                    verbose=False,
                )
                
                # --- Get data without output="dense" ---
                # This returns (n_pairs, 1), so we take [:, 0]
                con_data_upper = con.get_data()[:, 0]
                all_band_features.append(con_data_upper)

            # --- Flatten all features ---
            stacked_features = np.stack(all_band_features, axis=0)
            stacked_features_transposed = stacked_features.T
            features_flat = stacked_features_transposed.ravel()

            # --- DEBUG CHECK: Verify non-zero values ---
            mean_val = np.mean(features_flat)
            is_all_zero = np.all(features_flat == 0)
            # MODIFIED: Updated print statement for PLV
            print(f"    - [Debug Check] Mean PLV: {mean_val:.4f} | All zeros: {is_all_zero}")
            # --- End Debug Check ---
            
            features_list.append(features_flat)
            subject_ids.append(f"{group_name}_{subject_id}")

        except Exception as e:
            print(f"  ❌ ERROR processing subject {subject_id}: {e}")

    # --- Combine into DataFrame ---
    if not features_list:
        print(f"\nNo features were extracted for group {group_name}.")
        return

    # Create DataFrame with the correct column names and subject IDs
    features_df = pd.DataFrame(features_list, columns=column_names)
    features_df.insert(0, "subject_id", subject_ids)

    # --- Save to CSV ---
    # MODIFIED: Updated save path for PLV
    save_path = os.path.join(output_dir, f"{group_name}_plv_features.csv")
    features_df.to_csv(save_path, index=False)
    print(f"✅ Saved {group_name} features to {save_path}")
    print(f"    → Shape: {features_df.shape}")

# ----------- RUN FOR BOTH GROUPS -----------
# MODIFIED: Calls the renamed function
for grp in groups:
    extract_plv_features(grp)

# MODIFIED: Updated final print statement
print("\n🎯 All Phase Locking Value (PLV) features extracted and saved successfully.")


🔹 Processing group: ASD
  → Computing Phase Locking Value (PLV) for subject 001
    - Generating feature names (first subject only)...
    - Generated 9765 feature names.
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean PLV: 0.5124 | All zeros: False
  → Computing Phase Locking Value (PLV) for subject 002
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean PLV: 0.4551 | All zeros: False
  → Computing Phase Locking Value (PLV) for subject 003
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Com

In [2]:
!pip install mne_connectivity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 49.0 MB/s eta 0:00:00:00:01


In [4]:
# ============================================================
# EXTRACT SPECTRAL WEIGHTED PHASE LAG INDEX (WPLI) FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd
import mne
from mne_connectivity import spectral_connectivity_epochs

# ----------- PATH SETUP -----------
base_path = "/kaggle/input/preprocessed-small-autism-dataset/preprocessed"
# MODIFIED: Changed output directory name for WPLI features
output_dir = "/kaggle/working/wpli_features"
os.makedirs(output_dir, exist_ok=True)

# ----------- PARAMETERS -----------
groups = ["ASD", "TD"]
freq_bands = {
    "delta": (1.25, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}
# --- MODIFIED: Changed method from 'plv' to 'wpli' ---
method = "wpli"  # Weighted Phase Lag Index
band_names = list(freq_bands.keys())

# ----------- PROCESS FUNCTION -----------
# MODIFIED: Renamed function for clarity
def extract_wpli_features(group_name):
    group_path = os.path.join(base_path, group_name)
    fif_files = sorted([f for f in os.listdir(group_path) if f.endswith("-epo.fif")])

    features_list = []
    subject_ids = []
    column_names = None  # To store column names after first subject

    print(f"\n🔹 Processing group: {group_name}")
    for file in fif_files:
        subject_id = file.split("_")[0]
        file_path = os.path.join(group_path, file)
        # MODIFIED: Updated print statement for WPLI
        print(f"  → Computing Weighted Phase Lag Index (WPLI) for subject {subject_id}")

        try:
            # --- Load epochs ---
            epochs = mne.read_epochs(file_path, preload=True, verbose=False)
            ch_names = epochs.info["ch_names"]
            n_channels = len(ch_names)
            indices = np.triu_indices(n_channels, k=1) 

            # --- Generate Column Names (only once) ---
            if column_names is None:
                print("    - Generating feature names (first subject only)...")
                column_names = []
                for (idx1, idx2) in zip(indices[0], indices[1]):
                    pair_name = f"{ch_names[idx1]}-{ch_names[idx2]}"
                    for band_name in band_names:
                        column_names.append(f"{band_name}_{pair_name}")
                print(f"    - Generated {len(column_names)} feature names.")
            
            # --- Loop through each band one-by-one ---
            all_band_features = [] # To store features for each band
            
            for band_name, (fmin, fmax) in freq_bands.items():
                print(f"      - Computing {band_name} band ({fmin}-{fmax} Hz)...")
                con = spectral_connectivity_epochs(
                    epochs,
                    indices=indices, # Only compute the upper triangle
                    method=method,   # This will now use "wpli"
                    mode='multitaper',
                    fmin=fmin,
                    fmax=fmax,
                    sfreq=epochs.info['sfreq'],
                    faverage=True,
                    n_jobs=-1,
                    verbose=False,
                )
                
                # --- Get data without output="dense" ---
                # This returns (n_pairs, 1), so we take [:, 0]
                con_data_upper = con.get_data()[:, 0]
                all_band_features.append(con_data_upper)

            # --- Flatten all features ---
            stacked_features = np.stack(all_band_features, axis=0)
            stacked_features_transposed = stacked_features.T
            features_flat = stacked_features_transposed.ravel()

            # --- DEBUG CHECK: Verify non-zero values ---
            mean_val = np.mean(features_flat)
            is_all_zero = np.all(features_flat == 0)
            # MODIFIED: Updated print statement for WPLI
            print(f"    - [Debug Check] Mean WPLI: {mean_val:.4f} | All zeros: {is_all_zero}")
            # --- End Debug Check ---
            
            features_list.append(features_flat)
            subject_ids.append(f"{group_name}_{subject_id}")

        except Exception as e:
            print(f"  ❌ ERROR processing subject {subject_id}: {e}")

    # --- Combine into DataFrame ---
    if not features_list:
        print(f"\nNo features were extracted for group {group_name}.")
        return

    # Create DataFrame with the correct column names and subject IDs
    features_df = pd.DataFrame(features_list, columns=column_names)
    features_df.insert(0, "subject_id", subject_ids)

    # --- Save to CSV ---
    # MODIFIED: Updated save path for WPLI
    save_path = os.path.join(output_dir, f"{group_name}_wpli_features.csv")
    features_df.to_csv(save_path, index=False)
    print(f"✅ Saved {group_name} features to {save_path}")
    print(f"    → Shape: {features_df.shape}")

# ----------- RUN FOR BOTH GROUPS -----------
# MODIFIED: Calls the renamed function
for grp in groups:
    extract_wpli_features(grp)

# MODIFIED: Updated final print statement
print("\n🎯 All Weighted Phase Lag Index (WPLI) features extracted and saved successfully.")


🔹 Processing group: ASD
  → Computing Weighted Phase Lag Index (WPLI) for subject 001
    - Generating feature names (first subject only)...
    - Generated 9765 feature names.
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean WPLI: 0.1453 | All zeros: False
  → Computing Weighted Phase Lag Index (WPLI) for subject 002
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-30 Hz)...
      - Computing gamma band (30-45 Hz)...
    - [Debug Check] Mean WPLI: 0.1669 | All zeros: False
  → Computing Weighted Phase Lag Index (WPLI) for subject 003
      - Computing delta band (1.25-4 Hz)...
      - Computing theta band (4-8 Hz)...
      - Computing alpha band (8-13 Hz)...
      - Computing beta band (13-3